<a href="https://colab.research.google.com/github/John-588-git/Dashboard/blob/main/Image_segmentation6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install library
!pip install -q opencv-python matplotlib numpy scikit-image scikit-learn pillow
#import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage import segmentation, filters, color, morphology
from sklearn.cluster import KMeans
from PIL import Image
from google.colab import files

print("Libraries imported successfully!")

#Upload an image
uploaded = files.upload()

image_path = next(iter(uploaded.keys()))

image = cv2.imread("C:/Users/Administrator/Pictures/NACOA_2025.jpg")

if image is None:
    raise ValueError("Could not read the uploaded image.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Image loaded successfully:", image_path)

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

#4. Otsu's Thresholding
def otsu_segmentation(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    threshold, segmented = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return segmented, threshold


otsu_result, otsu_threshold = otsu_segmentation(image)

print("Otsu threshold:", otsu_threshold)

plt.figure(figsize=(8, 6))
plt.imshow(otsu_result, cmap="gray")
plt.title("Otsu's Thresholding")
plt.axis("off")
plt.show()
#5. Region Growing
def region_growing(image, threshold=15):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Smooth image to reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    h, w = gray.shape

    # Automatically select center as seed
    seed = (h // 2, w // 2)

    segmented = np.zeros_like(gray, dtype=np.uint8)
    visited = np.zeros_like(gray, dtype=bool)

    seed_value = int(gray[seed])

    stack = [seed]

    while stack:
        y, x = stack.pop()

        if y < 0 or y >= h or x < 0 or x >= w:
            continue

        if visited[y, x]:
            continue

        visited[y, x] = True

        if abs(int(gray[y, x]) - seed_value) <= threshold:
            segmented[y, x] = 255

            stack.extend([
                (y-1, x),
                (y+1, x),
                (y, x-1),
                (y, x+1)
            ])

    return segmented


region_result = region_growing(image)

plt.figure(figsize=(8, 6))
plt.imshow(region_result, cmap="gray")
plt.title("Region Growing")
plt.axis("off")
plt.show()
#6. K-Means Clustering

#Here, pixels are divided into 3 clusters based on their colour characteristics.
def kmeans_segmentation(image, k=3):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    pixels = rgb.reshape((-1, 3))
    pixels = np.float32(pixels)

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(pixels)
    centers = np.uint8(kmeans.cluster_centers_)

    segmented = centers[labels]
    segmented = segmented.reshape(rgb.shape)

    return segmented


kmeans_result = kmeans_segmentation(image, k=3)

plt.figure(figsize=(8, 6))
plt.imshow(kmeans_result)
plt.title("K-Means Segmentation (K=3)")
plt.axis("off")
plt.show()
#Experimenting with different values:
kmeans_result_4 = kmeans_segmentation(image, k=4)

plt.figure(figsize=(8, 6))
plt.imshow(kmeans_result_4)
plt.title("K-Means Segmentation (K=4)")
plt.axis("off")
plt.show()
#7. GrabCut Segmentation:requires an initial estimate of the foreground
def grabcut_segmentation(image):
    img = image.copy()

    mask = np.zeros(img.shape[:2], np.uint8)

    h, w = img.shape[:2]

    # Rectangle excluding a small border
    margin_x = max(5, int(w * 0.05))
    margin_y = max(5, int(h * 0.05))

    rect = (
        margin_x,
        margin_y,
        w - 2 * margin_x,
        h - 2 * margin_y
    )

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(
        img,
        mask,
        rect,
        bgd_model,
        fgd_model,
        5,
        cv2.GC_INIT_WITH_RECT
    )

    # Foreground and probable foreground
    foreground_mask = np.where(
        (mask == cv2.GC_FGD) |
        (mask == cv2.GC_PR_FGD),
        255,
        0
    ).astype("uint8")

    result = cv2.bitwise_and(
        cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
        mask=foreground_mask
    )

    return result, foreground_mask


grabcut_result, grabcut_mask = grabcut_segmentation(image)

plt.figure(figsize=(8, 6))
plt.imshow(grabcut_result)
plt.title("GrabCut Segmentation")
plt.axis("off")
plt.show()
#8. Watershed Segmentation
def watershed_segmentation(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Threshold
    _, binary = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Remove noise
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel,
        iterations=2
    )

    # Background
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    # Distance transform
    dist_transform = cv2.distanceTransform(
        opening,
        cv2.DIST_L2,
        5
    )

    # Foreground
    _, sure_fg = cv2.threshold(
        dist_transform,
        0.5 * dist_transform.max(),
        255,
        0
    )

    sure_fg = np.uint8(sure_fg)

    # Unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)

    # Markers
    _, markers = cv2.connectedComponents(sure_fg)

    markers = markers + 1
    markers[unknown == 255] = 0

    # Apply watershed
    img_copy = image.copy()
    markers = cv2.watershed(img_copy, markers)

    result = cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB)

    # Watershed boundaries
    result[markers == -1] = [255, 0, 0]

    return result, markers


watershed_result, watershed_markers = watershed_segmentation(image)

plt.figure(figsize=(8, 6))
plt.imshow(watershed_result)
plt.title("Watershed Segmentation")
plt.axis("off")
plt.show()
#9. Displaying all techniques side-by-side
#This is the main visualization required by the activity.
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original
axes[0, 0].imshow(image_rgb)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis("off")

# Otsu
axes[0, 1].imshow(otsu_result, cmap="gray")
axes[0, 1].set_title("Otsu Thresholding")
axes[0, 1].axis("off")

# Region Growing
axes[0, 2].imshow(region_result, cmap="gray")
axes[0, 2].set_title("Region Growing")
axes[0, 2].axis("off")

# K-Means
axes[1, 0].imshow(kmeans_result)
axes[1, 0].set_title("K-Means (K=3)")
axes[1, 0].axis("off")

# GrabCut
axes[1, 1].imshow(grabcut_result)
axes[1, 1].set_title("GrabCut")
axes[1, 1].axis("off")

# Watershed
axes[1, 2].imshow(watershed_result)
axes[1, 2].set_title("Watershed")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()


Libraries imported successfully!
